# indah - GRC map

A live demo of [**indah**](https://github.com/leejianrong/indah) - a reactive Python UI framework for cloud notebooks (no Node, single port, streaming over SSE).

The full app is laid out below - read it, tweak it, and re-run. **Runtime -> Run all**, then use the app that appears inline in the last cell.

In [ ]:
# Install indah from PyPI (plus any demo extras).
!pip install -q "indah" matplotlib

In [ ]:
# grc_map.py - the complete demo, inline (no clone, no download).
"""GRC map: Singapore's electoral divisions, colored by registered electors, over time.

A static choropleth per election year (2011, 2015, 2020, 2025) - pick a year and see
each Group Representation Constituency (GRC) / Single Member Constituency (SMC)
shaded by how many electors it has, on one fixed color scale so darker genuinely means
"more electors" across years, not just within one. A table alongside lists every
division's exact count, sortable.

Two layers, kept apart (ADR-0009), same shape as ``map_poster.py``:

- **The data layer** is offline: ``scripts/fetch_grc_map_data.py`` pulls the official
  electoral-boundary GeoJSON (data.gov.sg, Elections Department) for each year plus
  the matching "registered electors by constituency" table, joins them by division
  name, and writes ``examples/data/grc_map/<year>.json``.
  ``scripts/compile_grc_map_cache.py`` gzips + base64-embeds them into the ``_CACHE``
  block below. Re-run both after data.gov.sg publishes a new electoral-boundary year.
- **This module** only reads that embedded cache and draws it with matplotlib - no
  network call, no shapely import, at demo runtime. Embedding (rather than a sibling
  data file) is also what keeps this a single-file demo, portable into the
  auto-generated Colab notebook (``deploy/colab/make_colab.py`` inlines only the one
  named example file).

The boundary/electorate join isn't perfect: one 2025 division (Marine Parade-Braddell
Heights GRC) has no matching row in the electors table and renders as "no data" (a
flat neutral fill) rather than a guessed number - see the fetch script's docstring.

Data: Elections Department (ELD) via data.gov.sg, Singapore Open Data Licence.

Run it with:  python examples/grc_map.py   (prints a URL; embeds inline in a cell).
"""

from __future__ import annotations

import base64
import gzip
import json
from typing import Any

import indah

# BEGIN GENERATED CACHE (scripts/compile_grc_map_cache.py) --------------
_CACHE: dict[str, str] = {
    "2011": (
        "H4sIACz4rmoC/41cW4+cN3L9KwM92wbvl31r2wNpLGlGkGZgLAw/LBIjMJB4AW8QIFjsf09xWPXxdPehOk+SyxSbH1mXU9d/"
        "vvnf3/7255u/3L0Jzvs339y9+fff/+f3f/z+9z/+IcRf/vnmj7/912/jf/98/+X57oen05fnsei3//zt3/7773+ONT54l7LQ"
        "/vz9j/94/Ue//OJd/K6kFITsv4s5tvrrN3dKda1OaunxoMZYm1JDPqg+lkkNMRWj5t56H9Tguz/W1hDz66/5nPKxtubgw6S6"
        "eJyheT/P4KuP/qCGGCexleNgrcrJXn/MRQfUWuI8QkBqC3qw7BOsDXMHOWFf1JTDpPawdihZ18bWFjXW+PppoaSYFzWlubbk"
        "FBZVv0Ko+Gklt0kNCaipdF3b27qc3nRtbgmuLPhXanUtA3VuUEOC2011HqEm3Lb28kptrqxr8NEpNaa11vk4j9DSOm7tvaVJ"
        "lVcFqleqnHdR2+Sn0FOEtUUvstccFzXNx4zOl7qoNYZJjTEsaqxpUoW9F9UHXVsbUp1R148JO0z2h2uUf1Tmrt679Q1VeGBS"
        "5WQHtcjiKRM+rR1K8XNbkYnF/fKu8wS+ur6oqfZJLQ7WOm+yFtdxs95YlAdet5B9V6r83EFNUfcV7l3U2LruW5Bagq6Fu/G9"
        "eN0gJ6DawXJbRwhO/30HXgh+SnuUT1t3I585T5tcLEAN83ZTakDNce6bhHkX1ff5PtkvRdZFGudSv/RF6VWvJva+vkEUpNe1"
        "HqhV2Ua4fH2Z6ynpceGB3ZQp2TbABjnP08Yc2jqC3NikpvU6pZWqfNMXlxdRQ7qDX0uF85XoSlhUe3R5nkWVfXVtrAWpetwl"
        "Z6WUOE8gMgTUWKpSO2wbe726sCIPqdTsG1CnthAb0Y83E2WgHJL6kp6SvfP6kgFsUp+6KYLOFCnRC5O7BapqUuHQpc0vTN2v"
        "v/7rm7tlOZ9Pjz89Pb69+3R6e/p8aTzlO6u/Np6iqqZ9EBEroFKCPldIrYNOUe7yIm2gVKoKP4iNmE6V/QD8LXZUNVUP/ZZW"
        "43qR61Cub7lu7tm0e65oCbxZggBWwzezGmCikpo+NFBxcoxYhwpWR7VlaHIwNHz671uBXbueVnZY1BDMbsHS4NRGtgh2WtRW"
        "V8MH9jTkpNs6h3Z6CkOQl66wdpqHIO8Aa7tXiwzC0ORARU3y4hDZITS13hmperlFrBKcIU1kUqoLN7ACxxUcg2zwCsU28vf5"
        "axnRURVLONHR2lWAmOKopaqEv5To4dE3mI3hux5UzgR3Lq0mv3VAGAArNRkz+raopbesDFIWNYtBndS6DNSAM9FeHajeGd/A"
        "GZIqRhGIpdxbanre7pZYt+yMmuBqm0JM4WvYoE8VEkQTBthg6jr53PUNSY206wCTY1cN4oWFFzUHU0wop4rYotxyAUmdTDOs"
        "C8JOU3jAiy6rChJrvT7MiR3WH2tnVN3WwRHcAWBcR9iZk+lcoAbDH2cqs2fdAazDhda+sAM/vzt9/PR0urAAsjISA9CKYRmR"
        "l/VOAlrUCOPnFJ+MuoBTyyoaYs0X+pOXVnQgOACIakJFaSBf2gYV+F24SjfIsK04Lro2gEbMgkEUlgKz5KYYWJBZhU9Tx1AE"
        "FD7NOYO7eLLmDqSY4Rrwyi5u/9PT8zTCXx6ujLCYtMqeoCvAA10sOxtsjPBFimzEoIH0RBWf5PDbs+HDCnfaq0K2BkIpt3Os"
        "xcdWEy5r0y3GMDavoJ5FB3mTqfWAw3xeP6B4K/MBI1q/mgwgomYRjJoYtdjdLG/j7GovHuokr/Tx6e79w9PVM3XHsFJraoWS"
        "8wufN1HtdVIjSHP385DJoV7ueQJ8QZvLEnYnjPRKFRGPuMPr9clbgy1tZTqnYrmAp8Vbm2uTQFd47elkpODgUnNIeoYA0QPx"
        "1cI8L54huWlH5E8P6nYCjfHBSPT6waBBh3Pyev89g64U4Z+v0iP8lnhzRnXgzrf5Y/LWEdS1coCgTFCgcjl6goX7h2b3egRE"
        "ZlVlZvi58GPqq9UC8mU6XAw7EEXsJjWCvghuHiFWD3EVA3GxghvbBmjQHUAPiSrUfQNY4xBVD9UEQFKwUFJqQWoJui9IfpDX"
        "0jPAo4WiNqc6jztMHhNZ9HBejToNKuC1pipJcO+6XtHDUakLpgs36ReLdwN2vqofKh4acKm5wsUhgknKDbkgtVioL6MJU7+s"
        "dICBpYV2/RSDrMddPkwTztNncyvi08QudXug9cRNw2ziXEbYoagKFfCdQbAVgYg+KagXldE98HRTrNGFpxfRqSsrfwMFaGvH"
        "YQB3lsm9oiBBX1vIp8H3CmdkPUEG4BoOEcYfU++0J3j1mueri34ENH2uNS+U8Lunl7eiiC+xSsrFEQXcu0YnRKfmpT6TScHC"
        "Td1l4/fSgBqqUQGViFOpPIzekEAVlTmIq7Uepjckur6jrtaHFlcGqE4DQv8fZssaboPolRA1SiSMAs+vEc4oSCbAj5nI4doe"
        "9E0LhLrk5KpOCkh9tyCPOK1wN0V1V4kRqGcPcYmABvS5+/zw5dtPL49v3z59uLSvcnXMvnbhmmlJw1IPO+PIDOnG6HIDzY25"
        "7NvMssDnWjxIJKvdkO6NJuBaQwVGqGDMm/FdKvH6ecW7hDdL/jBjsNYMQAEHtitwEKOwmFFkR3+sAaIXqtrX5nGtU+g9INza"
        "t6neEvMAr+Y10twgzyLUoBY+B3hgVXEF5LRGwwKL6C38JOZw7elNeGtbhk3WVvuudbPdW4y49gXnezBHQ1TdOlYw50/EFBz2"
        "NsMpw7AdxMPa1RVi6SMoqsK/uENVqRjQ9UPCUmYr1wuIfVPzlxZ27uJ4mFGEW61mFGuC3y/dOK6tAHEv2d6lLiAhl2G31UBZ"
        "mgMu9mp9qyARA3/w3L6liT8dbusVXSTvlkqSB1NM6UvAp5k+sQD15e+ca4SrcOfHTw+P91+uIp319eWutEuJmqSAbJFIiprS"
        "DJFDUQLzZ0XDwu3nZEpvoQz+ekeUoi6VJQwRzJ6sKEePFnAXk4xUtRGi5oD5itqe7AKsrYp+UgKfYryEhrAh3iTMpP5QWWha"
        "Hrpb1mCFRIUF7bpABotTkJHdUk7nV3vxTPenbT5XVHu6fKj0nROkkuejRwOPQq0atxX3ztwhoYrFnm7LgR2FGDVmIq6zYXOh"
        "irczeSkeqRyhOs0gCN/l9WMuRjvCcg/E15j3lMIC7KL+ZhpTli7/ohuWkqUFbrTbUrj9rFoxhYQ6pXo9V1sISzSNUynpPd+S"
        "HS5nXCap/HJZp2qBaxCubbhm2mgxqvGKRuDFjMCncQF3eoQzhs01aO6rg4HjcrCRGSpf4sISWRTsVC1AA0oiumQJqYpawhJl"
        "YCMNxQUIb4ktOnKxcNyg2esobwo/FmaYWKgdTVS1dAwsrckCrx2/TPWU3CdQ4+SxEQwCQ931uA7vsaRkMXwL0Axhj0csZwlw"
        "M8yZQwWivk7pcclvs9jh8oEH1eJ2gsbiQZVPU/xx+MCDOoNfSUDc+jH5AhWII8Z0pZkuFN33D1/enR6/fX463X06/fXp3ZW6"
        "C6kTu8RjesUl5eUI3qsImVor8FO497vxlKlXzT3w5A3qJNhXtG82WJRuRAE2EYMU9R3B5Da5nWruCxA19DLcdsyU6Y9hsMtr"
        "9Ac/wSvTl9QhcmOZ1YIxLW9WtGQMHjWFB6UG+AJ3gECMah3ciQfQgLn4kFC4omwocBF+yqkwiomFAJrTOFVGLyWaXhVEUjEC"
        "p/UHFSpfBAEbJ0EwJ+Zmqg4+YQBTk0ZMnamTESEVfYQGRbOcJW00LBYw48kzBLscAw070xA1D2dvQt9WNSK2rN6InaNEXkj5"
        "59OPD493H0+XsFMss09EuoNWLjV0tXmud5MX5jlknm+mmWmaw+ZVUryiihZf0TotcaWspgvz1UlzlGeRR5r95ZlinlXmGWie"
        "reaZ7U0WnGbMeXadZ+LxzS/45+PpszgtYiGEke4vTUROr/U7V0xULWCePSiXqlY2B8wSRy2myBketlg1WM6gMEo0LwPj8GIa"
        "DEYljKYqEMP6CG6leOqKJbl4zmaT3+G5IJo34jkme5cBohtGea0mrkE0VlGBqIezOKGVr0AYqHetyRGoABGMAxl5CDU4Uzei"
        "4yGwkrV6BXTEBkR5w0uiDxc89E7hjst4hKKG1qeA8ZIjJwpE9XcESsLSIVmW/byFRS2wKnofztXVGoijF+FzrVAPoiAuVds2"
        "Y5gvmIKvGIK1wjVILrRejjosoFp0KJ1F41K6rs66kLILuf1y//H708/XoWq529fStmtYl6ZySSWBFUpuqsJUPfCgXPB0tATE"
        "giLymqirFRJ1TmNYYuRBnY6Iu+1QoIbKqAEKHY8UYskVqkA18ileIxZrBvWk67rUKu+mrnirsNZ39a8rFGzlrnlFrHmTtdG8"
        "7ggFo6FoqFf2gtLQal5zgILRrJ8GkdOak6ZMPWgDWap3Lr4krNVKFKHCCeKU8OSxJrKq4hBnYOHIOhILuhYKO1uYmnZQobzM"
        "aZQBU7kaSkke/L/a1ScbMQasf56SKHxb0TBPlk8JlarTJGbCl2w+ON2hYjFM0BiB2Dyoe1GxFXWHkMG+bNRMLmqeNkSoUCEx"
        "ilXmvuDiC8CZOZmUMd+Zrg8b8kT4wjQVDb5eeBSBAIRsO0AtbotahXq+r0ifXkLImKTWDAOYO0G3UVkcULNctJ4rQrhdJFHF"
        "AVNsAj+KCQlAU+X7hBtkRT0pOzhs0URCyhXyMedq5RJcPL08Pn9/en6+f7zMqIlQ0pIGav1q1yif63BM+UyzPZCgF1RogQE8"
        "fFdPQvR6RMtuhV4RqWZWQ0dsoCcTjAEn8912gLsW58xsJYCzahlV5yCXU4NZdgjDCmzSXo/eC1KTWuaGBX5hYg5Zi6glKkjt"
        "LeN5Fb2L5kCqIv3egGdLCfZr4OuJk6u/lrE4Z7K3+FM1YKpWkSvWXNFqQl55yGI0WyhCUcsG4DAwxIHTBmRRPHbOvJdZyIen"
        "x/v7yxIs0Sysh0gYWtGpW37gtkKeVtOzyntepb+r6I+aLY4N18ZiUcIIa7VmU1zyCJ0CIRQryC9wsmruO5RXn33wxdXJvd19"
        "eXp6vII6wlcsLb9R10yzUxPAjQU1LMJVRangDe8MFjNu3BBujCY1sNQYbww3s/EbNBAUTWDpkUuKXDI4aK5aljtDCYJTYRAq"
        "lBMJQtY4pgObScuneKVVsNI+qNjflHXxCjBWK8bLyngJGi9X46VtvAxuY0ip0d0YaGbLudnfQAQKJzjwoBCFwhkOfUaMQ1Ni"
        "GF89k9GrKpwPH06PP377/cv7h+e754ePp8vodR8ltET0N7xI2VaBwshH+a/j5Q225jicY3YK78VpME8gnlFVRhpuENR3FUQA"
        "voTlmLrz4M44V7WuADpFi7dSvo4uVbTCQQgk1VKPfD/2yqmc5rbcZzHnRoUwX63uyGgF8FGUcWOGoHJt1pwUoaJxUNV6QElk"
        "rZbRGiUm8D7OzEfI+GvWTAXthaUFCyrD3ShUiaPcDe5RDeMowISntJ40aOgYYFSPC1VQQtVrCLGgY6nhZ7xG2oe46Vnk/Y28"
        "F5L2TfIeS96PSVs3aZOnEJtGtOqNfizausXbvDYtYawRgTct7BoceDMEb5zgTRa0IcNbNVvwaOc3fR7BCvoxehK99Sw6jJ5o"
        "JnVAUewDNjGBoj6niD6O4iQ4r8UsEdG7oiE44X2PoCBYz2FCVKCwMoO3cuBpATYY049Wrwp1XzwFxZJVPK3FU2CbdBlPrdEs"
        "XLN0VzzrCVfGKWdI0ppwS+1nbfTNKibbjTyiFT8X6AgdaQX7MjSfm5JoWj69KbWmZdm8hDva60RXb5WGeysMPAsCtXWGWyXr"
        "tLqdV8LTqnleYS/4yTLo7SYePcMLFzDkp5fPT9cB1vFMhbhsUa1CyuDsRI2vxuwBIowqHk1/LKKIixKx5Zy1YO+6tXlnN+0C"
        "5x3jtLmc96HznvVoWRVsb+f98byXPmt0YfTdo8V1ZhtD+boV5faW2uaNHac2f4MPGJTgqGODUGqz1vAebyAfQ7HjCLcQVakK"
        "AIWVMHpuJysIIXu3lH7tCCyN75Y2q5P1X1EdNDanbnWAYZVF1OGkGuuvL7YC4dQBTKxKAcgpc4ES38lSdEB00QAkfG7SLIZc"
        "I2CMZHXH2MY0DqtNeA14YXy87gB2eOR51IompKq7mqC+tkb1zcdx1xmOSqkEzb9CtUAOjlCIln1MwCHRJDVl+C0rx0wRnuHs"
        "Fq/jsx8+3j88fvv+NByuS1U3GihIcGpTBMILRjg02MAIDjk4POFQhsEeDpE4nBrnDddlJGJ+4lXmcdPQy5t/eaMw7Snm3ce0"
        "UZk3NW/6n1mrNG+r5i3Ym3Zt2trN28CLVuOEUfmGcWYNM0N9Aw9Jb8LXPNS9CYvTEDoPt/PQ/CaMz0P+ND3AUwmbtANLUWzS"
        "GTT1UQ0WuALIttqkEof8tClJ4C2ytJ2Wdt7uKqusobq6cqvFmbZD89Zp3ma9acnm7dubVm/WFs5byHm3eVCHJvRcbhSuWSxu"
        "XBkWxFmkJGPcjpfJ8ZI6Xn5n9ZKCF26W9Z1p/0vI/PR098O7h9Nlcf1oOSVVCZvq4GAFtyI+/lb5h9ndAB+0KSChtSabshRa"
        "wrIpd6H5KF6LvKlb5jXOvB6alU6f3+PFo/zw7uUkr/Jy957Ui+RWBldeuTPC1xbiBUwm/6Uh3orIJWmDgyyFAVaq7cXGgzH2"
        "OuMsQUXeUHK6NC5eLT1NbS9AcimI0SqqIXFXYHaTVh2LllwSV6oF6wNEIQc4ttTIUhBlbDGDseAQj1lwGuMtS7ZK1l74M3e0"
        "ZPVd+ooyy0qFtg2yGJuxf3xEYLKO2REt3kxeujGliU90YsOf+JwoPlOKz5/azKric634DCw+L4uO1qJTuPjErs10LzYIjM8M"
        "O/pcxUKHWwlRnjxN/aj5hQSuc1bKmIH57XuhgKr0Yi0lBeXk8BCgcMdpne2YywDnUp2UwQMs3SrgAYyP1mNr+IOV5hBBmZT8"
        "vEWHAvjcovU06gTC75q3UnMMXGgSdQSdQH1URR3yJ7hOlpg4KyyL1jZXGwZf9GMrFPbVeLQ7Q2VgjfWo7K/oV6qYFwfOlyBp"
        "2wEcfGuoFT2CAwPtvqFwpSYrQ8zAXsOLVmqBqFCysWMZBy/aQASBJOmGc75x5KnTzwMEPJhQzedO0GG1SQaVateA89CEaoFY"
        "zHjwfBTPXdE8F8+JaapN9BCGm3iqbZOWo2V7Ws2Q5KVhh6SBSQEw6UblIK8y5BWJu+pFVum4qYqkFZQ5WqI6w+DRcwhw2V70"
        "mpv9dHr86RpUjHlMkWAKuRd7cYgEVusPF7yA9Z86PQL6zmrVuxrUCFRrGgcLWY9uePBEarEpHKN7Cznx6JwHnjPHqULHdZXn"
        "KNadDd9gDbYOs51nn3tZDzRnEdzdX3eijjAlnUeQVCeLK5++3ptOm9hptztrjN810bOG+11zvhXdi3cCNVq2NGNledAbxRjE"
        "xcdeVT8/vh1g9m5MZ750NQQhsr42PuaED1hoxVWbHJJuzBPho0f4lBI60YQPP6FzUvhMFT5/hY9qoVNdzq/mci7Vh59eHh/u"
        "f7yqvoqeOQ7bsQiMd/nTUy7ZMFRSh2yUc3x9tsRmDAWfWEGnW4y2FUuDhVtDMwyJV48tH3SED5/Lspnhwue98NkwbI4Mnzmz"
        "m09DR9lYkLV6iObQWTi7cS18tAudAsMHxtDhMnwQzWZoDZtvUyyRUT1SyzEmA0KGdJzTbvIT7VWybB0GIjctULRdqpqPkTMW"
        "3XZruPIu3ugw2XSj0M4V3uXCGmJ47wxvs+EdOSGosgiglHswjyJA7d+mQX3TzM4b32mT/Ga4BR2EwYdmBO3OjgVgmXxFtTKB"
        "EG/MfPEWBMSGwN0smc3cGVTGV2V2YkHfn97dPT59fr6ssAv1NdZ4nebO7vCq+g0HjDtr3LHjTqCzBOkoG/6qb8nd0I3HSn1b"
        "6gUTf3nnWtduDLMA585jJ849jwNUb7VeOZ1NfE+mESD7fWQQc0z4NkfA9uu5VZ6GpRlbeTmra8ClzC3deLDc26WeMfeiqcPN"
        "fXPux3OffxMfOOP6Czn668u7l6vxtvHVfF+LD0/S0nwuy/zyJDFPKPPk8yZRzZPaNAHOk+WbxDpPwvOEPcvtc049u0V5jl//"
        "9X+9gnCa0WUAAA=="
    ),
    "2015": (
        "H4sIACz4rmoC/41d24plR479laSebRP3S7+l7ZyqdLkyTVUWxhg/NDNmMMy4wT0MDE3/+ygylnasfY7Cp1+MUe3cO67SkrSk"
        "8483//frX/9485e7N8H5/OaLuzf/8dv//vb33/72+99F+PM/3vz+1//+dfzzu+fPb++f3o4nfv2vX//9f/72x3ggJNeriP74"
        "7ff/fP2Dn3/2Ln7Veq/hizv/VSylxl++uHuVdpdKfJXW7DxJW5nS4vqSho5naygqbT2V+WzpoS5pbPNr1dW0pMH1+Wzr9AZ9"
        "MsQlcy5Pqe/HuForeGupLSxpThhBo0dzxKdyWQNotePRQC/oDhM7PdvDHEGJLdO8nJ9ST8Kc8S1faVrFNbyAl+u0Db/88s8v"
        "7nhHn97evb9/d/f0/PHl3eXGNp/y9cbWmLFXuRzTr/LJOqW00jXUNAdaUz72tfqaPKSxLmnweEN0x75U15zuVmpLmvTZEI+v"
        "lV4bhKUvYfb4WCRhmH9eUkhLmOJ8MrdGfz4nm8vaqeqin1/Kbh1t+XzH38vwlrTguKWytrX0hmsgh3m91rk0hSnntS4ZL8gp"
        "rPUOPs0p5JzWaskhw7MxddobHIxM+1Xx/UzbFZufRzD1ugaQMNnUGwl1B7Jvaw9TmxPIIa7VSjIArGEiafcqzYGkeY5LhrVe"
        "m73uQm9rslnvdqn0gpZx3Ltb5yW5ilskN4/WQK9hy+tj467jDZmWhg/91S36/vv7p2+//Prz+8eXu5fHD/eXV8m7lMbhuFKS"
        "rtbX9yZX3NIPLr1uQ/LZLw3lQopTWugctFDrlDrahyr6TqVrDnJoptT1vlasiG7Cs2ntr6w0xuDXbag5B5Wu2zSkfr638Rvm"
        "HUuiWEk2z9e4KksYccC780tanMMV70vz1eIDTEKniyNTgI6gS1pqSbjkjh6tOKGNro1cXT23dL7EPhQcfL+WS+xDxi13tA0e"
        "alosWGDpfIOcsDWw2mLVZxttmcONdCHT19p8duzkGq9e89hpaUJvEPpOyzhPzfgYLXnH9R9KZT2bYNciGas6dOSrNEQ6CQUj"
        "CLyMDjohxEjSVOc99YX3QbWar45GW6CCvdhzmi7GJUqPlwaHyTeyLTXggIg58XwfCqQ0spawYuMWLmnLcwyu+TUGqHZXl8EY"
        "B3OOwPV1eUUKbetdpU/BCnta2VbxJV9ptj0Uj6Xt9K0CvRY8nfHeMYIQCIq4AI0dwrICTbBd0jc0kuI4C1KiZ5OPkKbKUqxi"
        "JYTiMtS7z3FJvQdECaTXvJwLHJC1jLLK2JwQ1t1pQU4mXrAsQROUgSOWFkgQacOEy1LZLURYmNADjcGnuQxietcYnCuw0plw"
        "lgBQD2mjJVNbkBrPoh5KKHqaW8oKS2kWMEZ9WckmK90UatIyNsAqsXGRpAem4xGUjDfwHHwDqkydh9Xx2ki7LkcBGjPXJQ06"
        "2tppd2JsCoEbPQpQUggayiIB1ojupT3LisFqoT0rHoDR0RyggAZwJJmHVDwKGpfCxcTrHbNiQDoJcmeK4kUaAe5O5SMqQq9D"
        "6Lw5+Fgsp1WE00Kn2eueixamfawqbJFvzpR23gWHSyLmlBbRlZIhpTc47G6Sq7mEJ6RxgWC+uxcAc/f1w6f7j1fIRS6iM5BL"
        "Ub1RF9htxeO4R5fX6uUS9SIuXS/KF+gr9MxSKKm2TJtAunh8bD2aHRYqZHqtHKug2oSGEHEyfV9uxPiYGiHaq5xxhgTkkFTR"
        "AIF7ecEcQuidXpBanMK2bPYYbZ/SZcPGk2EKHa9MmrZR1iDQt4DTRCeR0M3rLS9YiL1FaJ3xgjUCUVv6MdJQggAwrkD3KLp5"
        "hERKXwtt4pnzswGmXKSRpVia1jzduTbvd2hkSNvAR6Z0jmEcqyVNBW8okedWsWSRpa3hWU9aKvbpe4XaSZ8kGCyBPjTeFKYz"
        "IM+S/56Sw7N8GRPeylYszaBAOKmYFCYADGIUSJonJhMTl/jv5yqWnOmtE9nKh2h7U8QEZDvWoArU7ABvSyqYe05rzGBJa4eU"
        "rb64GPO9opf4WSyiD2SIa8TMHPnwYn0xMrmEa76CuosuYuORYSPJU5NjEXCgI61j0WtSl/vWxE/HyY0Ec4oe/caj9cdFLZmk"
        "Hi/ogaSikfAsQQxxuPEGMq41ToQhj5ICrxGLK6qCphvg5QzMRkPoqmsIPQlQTpBS4Et8HjwbSdtXNwcmzxLGKOpzy/7SewMs"
        "qRMAQdsDIO1KJ6nGLRwrvAbHUiAvbWVT38V3X0iaDP083GqFi53GSzbmwmQ9PTzcfXp+frq0VwMKd8NeDWP4av8SH9hYICQ/"
        "SJRfgNSTFRMV8LpMoocZLgATJwqutIGP8QJHUj81V0q8f3ARExtr+RIsuCAqgqP4lGCQysgVYQGyoi7A/S/+VlTBDkCYCGID"
        "NixgYoMYG+/Y0MhEUf14QWX/YcVm6jVGrnwvbCBnYz4bHtpQ0oadR5SSop82mLVhbwbsEaeHgW9U4MwTUy+78HUN0N3yrKP3"
        "NrgqAuk7wwNIC536CD0vCJcNGEbQE5s6DdhwBF0dSXFfKqGWgt1NZGjksgRNAhBECnBxa2EwFDSEfEIMRc8H+xRRAxhiUk5o"
        "aL5XThXpg4SlGf9OzyJQ0NgKi1GfE+6BPCNxqBJOfmNENoMCckvI1IhxxzVjPZ86rhnbnxHmxe0ltTmVpuAJGlZOiOcJPKgE"
        "gAsUimdMOn1OOae8kVm1RyaUl0pFmLAxgo4afuyVj0LVr5H1kXHNiQXWCalgDQJPV94wnxUQzsA4QdlGUiDJTTCTREhnAemU"
        "xHD9ZAMuDMoPn5/evn3+/u7h/tPLhVERgz7uxqVN6V4vQssrVCNIHFGOHimXhXPVC8lgjTttnwgRthTnkjI+DYkUudB1Peug"
        "+ZrnNwCEiXT5BF1MPt4bOPGWcT/lotN79VEKJAh0AChpFEm4WILLJX18FjN96VKKQzkCFVd5JYUsckco/9MRrkkcnXYV4UfZ"
        "3UBZIejm5H3itAx80t5JWhAXFRtJKZyILGAk91OkRWOoyyEcWArSRJZblhlR2EQh39PULhfp/tPjx7uPj5++xAm8RDQDOpqn"
        "L+D2isfABy3g7jUyp+JiJwTuyUCJ+kkw/oRJ5XvQSoygG6JLyfGNlPfiXHdOsDoYI9KATSNZcv7o7zWIO8ANpXihb8U+Ncpv"
        "IrtUWz+ljqO+wd867dZ92d0t6x5urqx1ty0l4DXgO2AEXaGi88okrcfK0Ax8O1aGng0aCalkiUT/xSN5Tc5YO+KMx4L3qAGW"
        "Sm5bTF2x0rLHXdHLQj9drnPUAN8hHAlTzbGuFSgevkahLIe44U3TxDTX0pHgk/1e7y2IK4sHGuljuLdyzGgJ24xniopagYmL"
        "a3N5GZ9fRkr89U5e3sORRTQDYdhpRvVFz1+i2Q8jDAUVCcvkiIQBh7QFPKjWqmSwFQDLRCnq5urxLGOD3vVZkupOiw+YbEfr"
        "z32yjf9WMV8+A01QBYbAQa+q5uVCCv2QKHRwXtyLvfp4/+3j092H+09XvIXuDH0pmiB4jRERGIa3L2vr2dFAQIpyyyKFty+6"
        "jMPu+gZKmovu1XASk0R8inhDZscQdAIZWOKEwhEf4ehyQpSoJlKNHjyXAUnZ4cSExf5knkVDSKmxV4FA4ikYP7daLorjxExE"
        "SKqxr5MRHxFHg3ydNn0KGVcjafcYV+bkQUZUqzY6hDIexMqY7SMQAmvuTiNrKqX5nrf94gx9evjw9f2P16Qmnybevb7tFbGB"
        "nGm3xZeY0nLychAFqJ4Qc4DJFW+XHeO5euI90twdlL08S7vigDtEw1AKvCNQmCpnHWExU8mVM83Td0q5EDmgKF7OgRLFVaML"
        "YmUpM56nakmZkoZVdUBKLMPEEmGUWtvcKJFS8rcpjk+caB6Bc+D4BeqqnAv4BzVS6jZO3JHEdFGaNU+fX6Tr3ssQ5hESaaMx"
        "IGIsQ/C0uLnAw6DQah0OHkI/mdK3EasQKcIghxf+V6SATh3BdfgzlIbvDms2MvlrZF19n+Apha3hm0h+3QjSqQvo642gkhV/"
        "2oWqasQG+8hJP4yWztgIR+jaENqzQ2jRYXXFiSx/HplLMBJpeJb/mqsnG+lvOZC2s2k7prYTazu8tnNsO9IZoNvn3G846LYv"
        "b7r9VoCgFtyHxHGWs2K7UpNPb9+Llrz78eHaOR4OgcX/9DDdjZPytsMgSKGqc8FBYRUGFgK/yoWgKHgB06ZVTnM0EBYaEZNa"
        "04i5/B9J8ag4CYnBzgQw3bMjcprZxVK93H/44fHp4dOVQRlMOcOLK8p+Invei7IpM8X4ZOk08hcIUmcggpH2/3NILtc2XZER"
        "xHCAWVI9Y38Nyrqli7tG+EojR0nQK4h8PZGbUbySIQO9oCqdieKkXXA1AHB2NFpl/4pRJWlRSoZfSZ+elb3B3lZBvFyGsFz8"
        "83pf7d3TdxP+v71OhMvks4UIkiZrQuYQqVO2TVu8vCYAG6yYTMdUAyuhMGyxOS0mK8Zm0GzYNhYxx+bwbPg+NjfI5hHZnCOL"
        "n2RzmXa8J5MjZfOpbOpVh3PkT0Pw+JjrgaUINskxpSEoq9XFSLZb6e1y9sjylogsfeUkT4OTMYjHtAzTQw6tMKroDdJMnqPz"
        "SOM2Nn0bd2Ljephuiu3S2O6P6SqZXtXGAbNRu43wN96A6TnYXobtkdjei+3p2F6R6T9VzdNnzi4ibV0SpfGGRoRXxpnIioNT"
        "CLpuMv0bUoDJHzCpBjYrwSQwWEwHmxNh8ydMroXNy9hwOGy+h80NsXkkJufE5qdsuCwm72XDp7G5N1XvZOdsUw8qPXmTSd/A"
        "dCmTFJRDUfrPEiq543Snd6yka/6STXUyWVE2gcomW4m/Cim5KgMNTyFHkAXKJyjy01lCnDEEHu3JKF+Y+QFi7755vs7zdAEz"
        "BtetyNHWKpClNEqMgC2ZIsTF41yJ1U3HnubeOkg0zGYXWP56vcVKUqmBaPgwpY7234NN4QWmeeZhTmErN1g4O8bOht0D1pCM"
        "kCgsaZrHEG4zjGzNtdFymIVIeWqm9rT0rB3R2kS/7EiZHVWzI3ARJDiZTftzU2yb7Z2Jt+GADR1smGFDEgQkZNgp3QI1Bsvd"
        "5sPb3HmNAvvAFRMZtRGCteotov5BUKLARYF3OAhKVBSUlApOFX+joqxrtYLnwiZg1kphHdERQJFxMR4Ge6EpRl+T0ELC0E9F"
        "WBiuaC8q2NJkxEigkRTzTYmibl6rBhMxW6s41KhHWauwy1ha2U07E7rLmm4yrEhudjoKLqM+LBJ0EYcMxyZSYrgIHIQGpkqu"
        "MpgwWLH1qBxbCN2yDOUotBn+yyEtmkYcyViWaj3LujulaF1QJOL5EOK9tbMU7lby/AaYx1EWUEkKlnNic5AbDkMi77hgH7Jf"
        "eLUgDidjXWepyGlGXVFYgaWSCiY2cuZLqoyWTPaxjKgaSgJX/aeAVXiBOa/rf2HRLuzjT5/ffb6/TKqEV5/guhhU/fwUF+wc"
        "FYe6butIRc37pG5ULCY+vwLDNE5AIVtRFdgjSkzXqMnDRGlKkWrpVUtcxacXlmuRUkJIKXBRWNJ0cyRmXZXlxFkliFnlpmv5"
        "V6D3Zi0gIz1gF16eVvGSnP/54/N1gkJUZLdYFOb7gyKVSKoyaHVLzhTb1yCvnDgqq7Q00k552YrOVIq2AjV1ra2WbRUetfRv"
        "WPkbpiErqhSTVW6YHNM6yTqC5hrImtvVaXYl26bqza6Qs6vp7Mo7q0bPruYzK/82VYJmRWEJR1KXizitQkW7pnFT/1hbuVLW"
        "m7pKJbiOIdyq1xTrggtXSW0VpyPjau/cu5ZPU0YnwwnnompR3VoxSmAt54Jr0mgjsxIrUyYFJY4mniUygNwuBH4TUXflhGq1"
        "dqCzbyqdjX6ydZmt90wdaevTje619bSp0239v7EVvahZoQro6A9rzDhQ2Q+J8fjGhrEivVDJH+4/Pj493P1w//H+24erSH8J"
        "A1FdBYprQ0BbNpFcrcPCM0N2sIuhm8n1KLVq0T89W7T1hWAfcsuOisbOBVrOH7wUEsIzT5GlSg08UVhMtovN3djwPGxOiM0f"
        "6U7NA3W0aFq8KHqpck8NDR937qkRVDFyT4x4hCLcqV4BUgoq72obGuyhuOU0sg4MR1X3I2+rQWUqr+ja7UPgN5P3QFBznuhd"
        "TglCzgembsIxLEzS7AC3jtMj8Xgt5ebF1lSt8KDETdTmHK73ExesamQ8EW2sqZFbrw39CNMswNmPGy3quFKWKKuPwOVCVQsD"
        "FwKRmSEmxOZMFkGTJMRHc0g+j8pCpkDqC0iDjQo79X4oodL6obIpm9G084zoYyp+Seka419c9kv18fz56eXr+5eXh6erZjp+"
        "WNcr3WGfN/HXsa+ds54FDp3j021X+WwqgszqoV2lkVmVZFcw2dVOdmWUXUVlF1yZtVl2HZdZ8yU6WKWcPC8OQ2CuT6k5acll"
        "ulGhtqlmsyrf7Co5sW0Jsa2wLOzmmpsaYaM8LEVjK6WNAjN13fmYXhnNb3549/Dx01XNlmgRbzVHsVW6rf43psIkK9qK3jYK"
        "GwNyGtk1OPj0+P3j09svf3r48e7l4QoguMmfunLdbAqW8jhTOnXwiEriyBRvTNWBoVMYyaDSWpQ/uXQRejpxKlmsswc7JRLg"
        "Fn+naIcXwoToTTVUJ+FHpZo79hqScsoDxZMGcNUqMZoEqtcExhOKFggLPjYp3zpYAKBi0OKkAgvCvahywsdaYxgf4PmImim3"
        "OtrY3W/MTjl2Vx27A4/drcfs7LMp9/Ngw/lKgLu36UenU4C0IRMx6qrSDcbYhl1mMtFs1tqG4Waz4WzmnM2yMxl5Nntvw/Sz"
        "WYE2g9BmG9rMRJPFaDMeN+xIi0dpMy5NduZZjVwop/vvv/v89Pjw7ZXTMlXldZHJpmTBKnyyK4TMYqJN3ZFVy2HXfYhUiwUp"
        "ZyQ6XIsFuYeettuTZ/N1b78RfqY2fvpscPWq+CVWX06WSFu+MLI3OwmaPQft7oR2J0Or66HdIdHupmj3Y9w1DbQbDJq9CM2u"
        "hWaDQ7sZ4qZxotVjsXTt5edZWo6iFEKKRdeWa1RFayAYRKhn4xNrFJPh58bVNt3yqrVkOTO0tt39ru6+Jwfa9iw2Xojpsdje"
        "jekJ2V6T7WDZvlg4oBJRO8VdhDcSiFM1sJbmgaiQKmpgkf5eDlbVwDF5ntpD7UQk3JAOTYKiyWUMyk8slX1Xu5DKLLqyC7Q2"
        "xVybwi/WuJfq++nt3Yfnu/ePz9dVgrVaINqu27Nr/Ox6QLN2cFdniBL/xMHDdtQhjIaZN3jIon+vUUvLIWEMgWgKqU61KjDN"
        "cz1wBljNVFMYAQ0c+2eCEzBhd6uCe1PtbVaG76rI7YpzszrdrGS3q943FfJ2Nf2m8t4q0rfr+a3Sf7NJgN1PQEwT9HLkpk3a"
        "ebUXphEdfUNDuqXCDWUv9r39yy19TVO+AwMWcLBBxgaQmNhFmc2VnVJNtjRmfe1qaS3C/SBowpvibsUmsd3mwJt0eZtav6Hh"
        "25R9i91/1lYXym/2V/36/uX5/WUEYTDaDY96l2RR007th5PSIMT1oxTJ0RiUMyR271urTa7dUdfM0Q63ByaNXVk7yWMmhOzk"
        "0XkNzCX94f7pu+sauDhHdLWoFZ5TLImShVUPdcmR/R4Fzo7yZNUowx4I9pBSqVnAx8QEUMa0axEwBT+KagHRpJzp04JhOtTy"
        "rPaRrdzxNaoWCJ1HG7Q4mt5wXoXLZX389O7+6cuX53tZ25+er9oBC0Kq2SxRRTpbVEy+0S7yKNBw+Ua7Sa+dcBw3/jG7WG46"
        "XprdMe1OmnbXzVBgTEIhrWsWPtg1EnY9xab2wqaE2vRRk2kKPO9L9TdaCG7aDW5aE1ptDM2Oh3ZzRLuR4qak2y7/NkvF7bLy"
        "TQm6Wa1uFrbbeUXTgG9svQULbASRtOVESYWp3047HaVwo//RpleSbo94mflWDya7/WnQXqstcJdQnDHRWnRTu+oUl250cKVm"
        "r9x3TF3pdKuH7FnVXOivb959vr+T/9y9N0qj5WBFw9MQVYxzODJFt6K5duTXDBJv4slm7NmOU5sxbTv+bcfKQ8oIqVWywqKD"
        "53gzqdDq0/Tl5Vl6r6sN5ZVM2etAEqMh0aLsNXQFFTNeiCVZNQhJ/WbK6G42peQuidpCnWxg5uMo7EDhqCM2Y0Uo1acFUQaT"
        "Gv5SWasuZ61et3UrGf19+zIvRX91YZSV3GDM2+z6DW/R5jjafEibO2nzLC1KpsXdtFmeNiN0wx61maY2K9VmsNpsV5MZa7No"
        "N4xbi5w7aByajfe3WiLZ7ZOSWiJGxuaPXZi/i7H7CY2DTdrrrV/mMH7EY/NzH+YPg5g/IWL82Ij9uySb3zCxf+/E/m0U+3dU"
        "7N9ciRq7FYRabv2qhfkLGPavZdi/rGH+CIf9ex32b3tsfgfE/s0Q6+dFNqS7eBBWKXVlU/ls2p9NEazKpEuNkkn2D0gU9XO4"
        "intI1Z5z0sb8CQv71y7MX8aw05tJeayNzdnZUl/Y/tH/blMd5a8r2NNXsnjIg8mZx76KtKKZvBgmPbIiLQh0ix5PhzCi4eHo"
        "K3EIA2oE0/o5C5E61MalcDTyHdKoHREo5yZbjZYohCbE5jd0f4rUNK1qAjVQGiWXro9S5DmjBZgoCG6gVb2Oq1OU2yMdL/7e"
        "rbZUdgsrs9uV3Rhr00TLbLhVkhrXQNkkuxWBgzN1qqLPNRza198ozt8U8ptF/3Zcf5MDsLIF6tXKzJjphp58eWkuGUC+5q6K"
        "aweAMaina7peacjO0cT01gfiuXbtPCK+X2Ap3CbRVpmOjda8EzQfjathyXnPolrnQE3xNokUm1dnU/BMup7J7NuQAG3CoE0u"
        "NImINpvJYD4NZZO1v2Jjqbp5OVQS9iNw4pc0RG3VqxsxpM0pXs1Lr8jc0O7Pl/UxEJ/GDxutj3WgRQGziV57Uo0XivbfHp7e"
        "jjDRZSwzvsbYrzLxCeQGOZmkNuxTbB74zd2wj1WpChDjiRuqhzj6G0d7cw3MK7O5XqcJy+L98s//B5vJMROrcAAA"
    ),
    "2020": (
        "..."
        "vO0AAA=="
    ),
    "2025": (
        "..."
    ),
}
# END GENERATED CACHE -----------------------------------------------------

YEARS: list[str] = sorted(_CACHE)
DEFAULT_YEAR = YEARS[-1] if YEARS else ""

NO_DATA_COLOR = "#d8d2c8"  # a neutral Studio muted tone
INK = "#2b2320"
RAMP_LOW = "#eef5f2"  # pale teal tint
RAMP_HIGH = "#173d35"  # deep teal, darker than the Studio accent for more range

_decoded: dict[str, dict[str, Any]] = {}


def _load(year: str) -> dict[str, Any]:
    """Decode one year's divisions, lazily and once (gzip+base64 -> JSON)."""
    if year not in _decoded:
        raw = gzip.decompress(base64.b64decode(_CACHE[year]))
        _decoded[year] = json.loads(raw)
    return _decoded[year]


def _global_domain() -> tuple[int, int]:
    """The min/max registered-electors count across every cached year, so the color
    scale is comparable across years, not renormalized per year."""
    values = [
        d["electors"]
        for year in YEARS
        for d in _load(year)["divisions"]
        if d["electors"] is not None
    ]
    return (min(values), max(values)) if values else (0, 1)


def _lerp_color(t: float) -> tuple[float, float, float]:
    def hex_to_rgb(h: str) -> tuple[float, float, float]:
        h = h.lstrip("#")
        return tuple(int(h[i : i + 2], 16) / 255 for i in (0, 2, 4))

    lo, hi = hex_to_rgb(RAMP_LOW), hex_to_rgb(RAMP_HIGH)
    return tuple(lo[i] + (hi[i] - lo[i]) * t for i in range(3))


def render_choropleth(year: str):
    """Render one year's divisions as a styled matplotlib Figure, colored by
    registered electors on the fixed global scale."""
    import matplotlib

    matplotlib.use("Agg")  # a server has no display; also skips pyplot's global state
    import math

    from matplotlib.cm import ScalarMappable
    from matplotlib.collections import PolyCollection
    from matplotlib.colors import LinearSegmentedColormap, Normalize
    from matplotlib.figure import Figure

    data = _load(year)
    vmin, vmax = _global_domain()
    cmap = LinearSegmentedColormap.from_list("indah_teal", [RAMP_LOW, RAMP_HIGH])
    norm = Normalize(vmin=vmin, vmax=vmax)

    rings: list[list[list[float]]] = []
    colors: list[tuple[float, float, float]] = []
    all_lat: list[float] = []
    for division in data["divisions"]:
        count = division["electors"]
        color = _lerp_color(norm(count)) if count is not None else _lerp_color(-1)
        for ring in division["rings"]:
            rings.append(ring)
            colors.append(color if count is not None else None)
            all_lat.extend(y for _x, y in ring)
    # `None` placeholders (no-data divisions) render as the neutral gray, not the ramp.
    face_colors = [c if c is not None else NO_DATA_COLOR for c in colors]

    lat0 = sum(all_lat) / len(all_lat) if all_lat else 1.35

    fig = Figure(figsize=(6, 4.6), dpi=150)
    ax = fig.add_subplot(111)
    fig.patch.set_facecolor("#fdf6ee")
    ax.set_facecolor("#fdf6ee")
    coll = PolyCollection(rings, facecolors=face_colors, edgecolors=INK, linewidths=0.5)
    ax.add_collection(coll)
    ax.autoscale_view()
    ax.set_aspect(1 / math.cos(math.radians(lat0)))
    ax.axis("off")
    ax.margins(0.02)

    sm = ScalarMappable(norm=norm, cmap=cmap)
    cbar = fig.colorbar(sm, ax=ax, orientation="horizontal", fraction=0.05, pad=0.03, shrink=0.6)
    cbar.set_label("Registered electors", fontsize=8, color=INK)
    cbar.ax.tick_params(labelsize=7, color=INK, labelcolor=INK)

    ax.set_title(f"Singapore electoral divisions - {year}", fontsize=12, color=INK, pad=10)
    fig.text(
        0.5,
        0.02,
        "Data: Elections Department (ELD), data.gov.sg",
        ha="center",
        fontsize=6.5,
        color=INK,
        alpha=0.7,
    )
    fig.subplots_adjust(left=0.02, right=0.98, top=0.9, bottom=0.12)
    return fig


def _stats(year: str) -> dict[str, Any]:
    divisions = _load(year)["divisions"]
    with_data = [d for d in divisions if d["electors"] is not None]
    total = sum(d["electors"] for d in with_data)
    biggest = max(with_data, key=lambda d: d["electors"]) if with_data else None
    return {"total": total, "count": len(divisions), "biggest": biggest}


def _table_source(year: str) -> dict:
    rows = sorted(
        _load(year)["divisions"],
        key=lambda d: (d["electors"] is None, -(d["electors"] or 0)),
    )
    return {
        "columns": ["Division", "Registered electors"],
        "rows": [
            [d["name"], d["electors"] if d["electors"] is not None else "no data"] for d in rows
        ],
    }


def build() -> indah.Session:
    year = indah.Signal(DEFAULT_YEAR)

    intro = indah.Text(
        "# Singapore electoral divisions over time\n\n"
        "Pick a year to see each GRC/SMC shaded by its registered electors, on one "
        "fixed scale so darker means the same thing across every year. The table "
        "below lists the exact counts.",
        markdown=True,
    )

    year_radio = indah.Radio(year, options=YEARS, label="Election year")

    chart = indah.Plot(lambda: render_choropleth(year.value), alt="electoral divisions choropleth")

    stats = indah.Row(
        children=[
            indah.Stat(value=lambda: f"{_stats(year.value)['total']:,}", label="Total electors"),
            indah.Stat(value=lambda: str(_stats(year.value)["count"]), label="Divisions"),
            indah.Stat(
                value=lambda: (_stats(year.value)["biggest"] or {}).get("name", "-"),
                label="Largest division",
            ),
        ]
    )

    table = indah.Table(lambda: _table_source(year.value), label="Every division")

    page = indah.Column(
        children=[
            intro,
            indah.Card(children=[year_radio]),
            indah.Card(children=[chart]),
            indah.Card(children=[stats]),
            indah.Card(children=[table]),
        ]
    )
    return indah.Session(page)


app = indah.create_app(session_factory=build)

In [ ]:
# Launch it - in Colab this embeds the app inline in the cell output.
import indah
indah.launch(app)